In [2]:
import time
from datetime import datetime, timedelta

import pandas as pd
import requests

LATITUDE = 57.505
LONGITUDE = -3.077

MODELS = ["ecmwf_ifs"]            
VARIABLES = ["temperature_2m", "wind_speed_10m", "wind_speed_80m", "wind_direction_10m", "wind_direction_80m"]
RUN_HOURS = [0, 12]
MAX_LEAD = 48
BASE_URL = "https://single-runs-api.open-meteo.com/v1/forecast"

start_date = datetime(2024, 4, 1)
end_date = datetime(2026, 4, 30)   

session = requests.Session()
all_runs = []

for model in MODELS:
    day = start_date
    while day <= end_date:
        for h in RUN_HOURS:
            run_dt = day.replace(hour=h)
            run_str = run_dt.strftime("%Y-%m-%dT%H:%M")

            params = {
                "latitude": LATITUDE,
                "longitude": LONGITUDE,
                "run": run_str,
                "models": model,                 
                "hourly": ",".join(VARIABLES),
                "forecast_days": 3,              
                "timezone": "GMT",
                "wind_speed_unit": "ms", 
            }

            try:
                r = session.get(BASE_URL, params=params, timeout=60)
                if r.status_code != 200:
                    try:
                        reason = r.json().get("reason", "")
                    except ValueError:
                        reason = r.text[:200]
                    print(f" {model} {run_str}: HTTP {r.status_code} – {reason}")
                else:
                    hourly = r.json().get("hourly", {})
                    df = pd.DataFrame({"forecast_time": pd.to_datetime(hourly["time"])})
                    for var in VARIABLES:
                        df[var] = hourly.get(var)

                    df["fcst_step"] = ((df["forecast_time"] - run_dt) / pd.Timedelta(hours=1)).astype(int)
                    df = df[(df["fcst_step"] >= 0) & (df["fcst_step"] <= MAX_LEAD)].copy()

                    if len(df) != MAX_LEAD + 1:
                        print(f" {model} {run_str}: {len(df)} sati umesto {MAX_LEAD + 1}")

                    df.insert(0, "model", model)
                    df.insert(1, "run_init", run_dt)
                    all_runs.append(df)

            except Exception as e:
                print(f"{model} {run_str}: {e}")

            time.sleep(0.2)

        day += timedelta(days=1)

if all_runs:
    final_df = pd.concat(all_runs, ignore_index=True)
    fname = f"open_meteo_single_runs.csv"
    final_df.to_csv("../data/nwp/%s" %fname, index=False)
    print(f"Sačuvano {len(final_df)} redova u {fname}")
else:
    print("Ništa nije preuzeto.")

 ecmwf_ifs 2025-08-05T00:00: HTTP 400 – The requested model run is not available. Model: ecmwf_ifs, run: 2025-08-05T00:00Z
 ecmwf_ifs 2025-08-06T00:00: HTTP 400 – The requested model run is not available. Model: ecmwf_ifs, run: 2025-08-06T00:00Z
 ecmwf_ifs 2025-08-08T00:00: HTTP 400 – The requested model run is not available. Model: ecmwf_ifs, run: 2025-08-08T00:00Z
 ecmwf_ifs 2025-08-08T12:00: HTTP 400 – The requested model run is not available. Model: ecmwf_ifs, run: 2025-08-08T12:00Z
 ecmwf_ifs 2025-08-09T00:00: HTTP 400 – The requested model run is not available. Model: ecmwf_ifs, run: 2025-08-09T00:00Z
Sačuvano 74235 redova u open_meteo_single_runs.csv


In [9]:
final_df

,model,run_init,forecast_time,temperature_2m,wind_speed_10m,wind_speed_80m,wind_direction_10m,wind_direction_80m,fcst_step
0,ecmwf_ifs,2024-06-01 00:00:00,2024-06-01 00:00:00,8.8,3.13,5.54,299,307,0
1,ecmwf_ifs,2024-06-01 00:00:00,2024-06-01 01:00:00,8.5,3.24,5.83,295,302,1
2,ecmwf_ifs,2024-06-01 00:00:00,2024-06-01 02:00:00,8.0,3.12,5.78,292,302,2
3,ecmwf_ifs,2024-06-01 00:00:00,2024-06-01 03:00:00,7.9,3.15,5.75,285,296,3
4,ecmwf_ifs,2024-06-01 00:00:00,2024-06-01 04:00:00,7.7,3.04,5.55,279,290,4
...,...,...,...,...,...,...,...,...,...
485,ecmwf_ifs,2024-06-05 12:00:00,2024-06-07 08:00:00,8.9,5.27,7.35,237,237,44
486,ecmwf_ifs,2024-06-05 12:00:00,2024-06-07 09:00:00,9.2,5.19,7.17,238,237,45
487,ecmwf_ifs,2024-06-05 12:00:00,2024-06-07 10:00:00,9.9,4.83,6.52,243,243,46
488,ecmwf_ifs,2024-06-05 12:00:00,2024-06-07 11:00:00,11.0,4.63,6.11,263,262,47
